# POC · Can a model identify room contents and how much space they take?

> **FILE PURPOSE** — The implementation. Nine stages that take a room video and
> return a volume, two competing methods raced against each other, plus an
> appendix that needs no data at all.
>
> Read `../ARCHITECTURE.md` first for the diagram and how the stages connect.
> Open decisions: `../GAPS.md` · Build state: `../STATUS.md`

**Goal.** Point a camera at a room. Get back a list of what is in it and how many cubic
metres it will take on a removal truck. Then find out how wrong that answer is.

## The two methods, in plain terms

We do not know the best way to get from a photo to a volume, so we build both and race them.

### Method A · Recognise & Look Up
Works like **a surveyor with a clipboard.** Look at the sofa, decide *"that's a 3-seater"*,
then read the volume off a standard table: `sofa_3_seat = 1.42 m³`.
**It never measures anything.** The AI's only job is to name things correctly.

### Method B · Measure & Compute
Works like **a surveyor with a tape measure.** Work out the sofa is 2.1 m x 0.9 m x 0.85 m,
then calculate the volume from those numbers.
**It never looks anything up.** The AI's only job is to measure accurately.

### Why race them?
They fail for different reasons. Method A is only as good as the lookup table and the
naming. Method B is only as good as the measuring — and a photo has no built-in sense of
scale, which is the hard part. Every product on the market today uses Method A and none of
them measure objects, but we have not verified that on our own footage. One test, then we
drop the loser and stop paying for it.

> **Without `ground_truth.csv` the numbers here are meaningless.** You need hand
> measurements of the same rooms to compare against. See gap **A1** in `../GAPS.md`.

**Read `ARCHITECTURE.md` first** if you want the diagram of how the stages fit together.

## Stage 0 · Configuration — every knob in one place

In [ ]:
# ============================================================================
# STAGE 0a · PATHS AND SETTINGS
#
# WHY THIS EXISTS
#   Every threshold, model name and switch lives here so an experiment is a one-line
#   edit, not a hunt through the notebook. The two switches that matter most are
#   `use_scale_anchor` (stage 4) and `dedup_rule` (stage 7).
#
# IN   nothing
# OUT  CFG dict, CUBE lookup table, VOCAB detector prompts
#
# HOW IT HELPS THE GOAL
#   The POC's real output is a comparison. Comparisons need one variable changed at a
#   time, which needs all the variables in one visible place.
# ============================================================================

from pathlib import Path
import json, os, sys

# Work whether the notebook is launched from the repo root or from inside poc/
ROOT = Path.cwd() if (Path.cwd() / "cube_table.json").exists() else Path.cwd() / "poc"
assert (ROOT / "cube_table.json").exists(), f"cube_table.json not found relative to {Path.cwd()}"
IN_DIR, OUT_DIR = ROOT / "data" / "input", ROOT / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    # ---- stage 1 · which frames we keep -------------------------------------
    frame_stride_s   = 1.0,      # take one frame per N seconds of video
    max_frames       = 16,       # hard cap: more frames = more cost, diminishing returns
    blur_min         = 60.0,     # variance-of-Laplacian floor. Below this = too blurry.
    bright_range     = (35, 225),# reject near-black and blown-out frames
    resize_long_edge = 1024,     # models don't need more; smaller = faster

    # ---- stage 2 · object detection -----------------------------------------
    det_model  = "IDEA-Research/grounding-dino-base",
    det_box_th = 0.30,           # lower = finds more, invents more
    det_txt_th = 0.25,           # how confidently a box must match the text prompt

    # ---- stage 3 · depth ----------------------------------------------------
    # MoGe-2 is MIT licensed so we can ship it. UniDepthV2 scores better indoors
    # but is CC BY-NC (non-commercial) — see gap B2. Do not swap it in.
    depth_model = "Ruicheng/moge-2-vitl",   # smaller/faster: moge-2-vitb-normal, moge-2-vits-normal

    # ---- stage 4 · scale anchor ---- THE MOST IMPORTANT SWITCH IN THE FILE ---
    # A photo has no sense of size. This corrects it using a door of known height.
    # Run the notebook twice, True then False, and compare stage 9. That difference
    # is the single most valuable number this POC produces. See gap A5.
    use_scale_anchor = True,
    door_height_m    = 1.981,    # standard UK internal door leaf
    manual_scale     = None,     # set a float to force a factor and skip detection

    # ---- stage 6 · Method A (Recognise & Look Up) ---------------------------
    vlm_model      = "claude-sonnet-5",
    vlm_max_frames = 6,          # cost control, roughly $0.02 per frame

    # ---- stage 7 · de-duplication -------------------------------------------
    # "max"    = trust the frame that saw the most (survives things being hidden)
    # "median" = trust the typical frame (survives the model imagining things)
    dedup_rule = "max",

    # ---- stage 9 · scoring --------------------------------------------------
    score_room = None,           # room_id from ground_truth.csv, or None for the first
)

# The lookup table that turns a NAME into a VOLUME. This is Method A's entire brain.
CUBE    = json.loads((ROOT / "cube_table.json").read_text())
CLASSES = CUBE["classes"]
CLASS_NAMES = sorted(CLASSES.keys())

# The words we hand the detector, each mapped to the size classes it could turn out to be.
_v      = json.loads((ROOT / "detect_vocab.json").read_text())
VOCAB   = _v["prompts"]

print(f"root          {ROOT}")
print(f"size classes  {len(CLASS_NAMES)}")
print(f"  table status: {CUBE['_meta']['status']}")
print(f"detector words {len(VOCAB)}")
print(f"input files   {sorted(p.name for p in IN_DIR.iterdir() if p.name != '.gitkeep') or 'NONE — add a video or photo'}")

In [ ]:
# ============================================================================
# STAGE 0b · LIBRARIES AND HARDWARE
#
# WHY THIS EXISTS
#   The two AI models need a compute device. Macs have no CUDA, so we fall back to
#   Apple's "mps" GPU backend, then to CPU. Method A needs an API key; if it is
#   missing we skip that method rather than crashing, so Method B can still run.
#
# IN   nothing
# OUT  DEVICE (where models run), HAS_KEY (whether Method A is available)
#
# HOW IT HELPS THE GOAL
#   Lets someone run half the experiment with no API key and no GPU, which lowers
#   the barrier to getting a first number.
# ============================================================================

import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

def pick_device():
    """Best available compute device: NVIDIA GPU > Apple GPU > CPU.

    Override with:  export NX_DEVICE=cpu
    Needed occasionally on Apple Silicon when an op has no Metal kernel.
    """
    forced = os.environ.get("NX_DEVICE", "").strip().lower()
    if forced in ("cpu", "cuda", "mps"):
        return torch.device(forced)
    if torch.cuda.is_available():         return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

DEVICE  = pick_device()
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))

print(f"torch {torch.__version__} running on {DEVICE}")
print(f"Method A (Recognise & Look Up): {'ENABLED' if HAS_KEY else 'SKIPPED — no ANTHROPIC_API_KEY'}")
print(f"Method B (Measure & Compute):   ENABLED")
if DEVICE.type == "cpu":
    print("\n! CPU only — MoGe-2 vitl will take ~30s per frame.")
    print("  Switch CFG['depth_model'] to 'Ruicheng/moge-2-vits-normal' to speed it up.")

## Stage 1 · Ingest — from an uploaded file to usable frames

**Input:** one video or photo. **Output:** ~16 sharp still frames.

A one-minute video is about 1,800 frames and most are useless — blurred by hand shake,
showing the same wall twice, or aimed at the carpet. We sample evenly, then throw away
anything too blurry or too dark to work with.

The rejection count is printed on purpose. **If most of your footage is being thrown
away, the filming instructions need fixing, not the threshold.**

In [ ]:
# ============================================================================
# STAGE 1 · INGEST — whatever was uploaded becomes a list of good still frames
#
# WHY THIS EXISTS
#   Both AI models work on single images. Video has to become frames first, and
#   sending blurry frames wastes money and produces bad boxes AND bad depth.
#
# IN   one file from poc/data/input/ — .mp4/.mov video or .jpg/.png image
# OUT  `frames` = list of dicts, each {t, img, sharp, bright, ok}
#         t      seconds into the video
#         img    the picture itself, as a BGR numpy array
#         sharp  sharpness score (higher = crisper)
#         bright average brightness 0-255
#
# HOW IT HELPS THE GOAL
#   Frame quality is the ceiling on everything downstream. This is the cheapest
#   place in the whole pipeline to improve the final answer.
# ============================================================================

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
VID_EXT = {".mp4", ".mov", ".m4v", ".avi", ".mkv"}

def resize_long(img, long_edge):
    """Shrink so the longest side is `long_edge`. Never enlarges."""
    h, w = img.shape[:2]
    s = long_edge / max(h, w)
    return cv2.resize(img, (int(round(w*s)), int(round(h*s))), interpolation=cv2.INTER_AREA) if s < 1 else img

def sharpness(gray):
    """Variance of the Laplacian: a standard blur score. Flat/blurry image -> low value."""
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def load_keyframes(path, cfg):
    """Read an image or video and return (every_sampled_frame, the_good_ones)."""
    raw = []
    if path.suffix.lower() in IMG_EXT:
        img = cv2.imread(str(path))
        if img is None: raise IOError(f"could not read {path}")
        raw = [(0.0, img)]                      # a photo is just a 1-frame video
    elif path.suffix.lower() in VID_EXT:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        stride = max(1, int(round(fps * cfg["frame_stride_s"])))   # frames between samples
        i = 0
        while True:
            ok, fr = cap.read()
            if not ok: break
            if i % stride == 0: raw.append((i / fps, fr))
            i += 1
        cap.release()
    else:
        raise ValueError(f"unsupported file type: {path.suffix}")

    # Score every sampled frame, then keep only the ones that pass both gates.
    out = []
    for t, fr in raw:
        fr = resize_long(fr, cfg["resize_long_edge"])
        g  = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
        s, b = sharpness(g), float(g.mean())
        out.append(dict(t=t, img=fr, sharp=s, bright=b,
                        ok=(s >= cfg["blur_min"] and cfg["bright_range"][0] <= b <= cfg["bright_range"][1])))
    kept = [f for f in out if f["ok"]][: cfg["max_frames"]]
    return out, kept

inputs = sorted(p for p in IN_DIR.iterdir() if p.suffix.lower() in (IMG_EXT | VID_EXT))
if not inputs:
    raise SystemExit(f"No input files. Put a room photo or video in {IN_DIR}")

TARGET = inputs[0]                    # change the index to process a different file
sampled, frames = load_keyframes(TARGET, CFG)

print(f"{TARGET.name}")
print(f"  sampled {len(sampled)} frames, kept {len(frames)}, rejected {len(sampled)-len(frames)}")
if sampled:
    print(f"  sharpness seen: {min(f['sharp'] for f in sampled):.0f} to {max(f['sharp'] for f in sampled):.0f}"
          f"   (rejecting below {CFG['blur_min']})")
    if len(frames) < 3 and len(sampled) > 5:
        print("  ! Very few frames survived. Either the footage is shaky or blur_min is too strict.")

In [ ]:
# ============================================================================
# STAGE 1 CHECK · look at what survived
# Purely visual. If these frames don't cover the room, no model can fix that later —
# missing coverage is a bigger error source than model accuracy (gap C2).
# ============================================================================

n = min(len(frames), 8)
if n:
    fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
    for ax in axes.ravel(): ax.axis("off")
    for ax, f in zip(axes.ravel(), frames[:n]):
        ax.imshow(cv2.cvtColor(f["img"], cv2.COLOR_BGR2RGB))
        ax.set_title(f"t={f['t']:.1f}s  sharp={f['sharp']:.0f}", fontsize=9)
    plt.suptitle(f"Stage 1 — frames kept from {TARGET.name}", fontsize=11)
    plt.tight_layout(); plt.show()

## Stage 2 · Detection — find and count the objects

**Input:** the keyframes. **Output:** a labelled box around every object, per frame.
**Model:** Grounding DINO — an "open-vocabulary" detector, meaning you tell it what to look
for in plain words rather than retraining it.

Both methods use these boxes, for different reasons:
- **Method B** measures what is inside each box.
- **Method A** uses the label to narrow down which size classes are even possible.

**Why the detector counts and not the AI chat model:** measured counting accuracy for
vision language models is about 0.53, and it degrades further with several object types in
frame. Worse, it under-counts — producing confident, plausible, *low* inventories, which
become low quotes and undersized trucks. So: **the detector counts, the chat model names.**

`door` is in the word list but excluded from the inventory. It is there only to act as the
measuring stick in stage 4.

In [ ]:
# ============================================================================
# STAGE 2 · DETECTION — draw a labelled box around every object we care about
#
# WHY THIS EXISTS
#   We need to know WHERE things are (so Method B can measure them) and HOW MANY
#   there are. Grounding DINO takes a list of words and finds those things, so we
#   can change what we look for by editing detect_vocab.json — no retraining.
#
# IN   frames from stage 1
# OUT  f["dets"] on each frame = list of {label, score, box}
#         label  which word it matched, e.g. "sofa"
#         score  0-1 confidence
#         box    [x0, y0, x1, y1] in pixels
#
# HOW IT HELPS THE GOAL
#   This is the counting mechanism for the whole POC, and the anchor points for
#   measurement. Anything the vocabulary misses is invisible to both methods —
#   which is why stage 9 prints a "missed entirely" list.
# ============================================================================

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image

_det = {}
def det_load():
    """Load the detector once and cache it — it is slow to initialise."""
    if "m" not in _det:
        print(f"loading {CFG['det_model']} (first run downloads ~700MB) ...")
        _det["p"] = AutoProcessor.from_pretrained(CFG["det_model"])
        _det["m"] = AutoModelForZeroShotObjectDetection.from_pretrained(CFG["det_model"]).to(DEVICE).eval()
    return _det["p"], _det["m"]

# Grounding DINO expects lowercase phrases joined by ". " with a trailing period.
DET_PROMPT = ". ".join(VOCAB.keys()).lower() + "."

def detect(img_bgr):
    """Run the detector on one frame. Returns a list of labelled boxes."""
    proc, model = det_load()
    pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    inp = proc(images=pil, text=DET_PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inp)
    kw = dict(input_ids=inp["input_ids"], target_sizes=[pil.size[::-1]],
              text_threshold=CFG["det_txt_th"])
    # transformers renamed this argument around v4.51 — support both spellings.
    try:
        res = proc.post_process_grounded_object_detection(out, threshold=CFG["det_box_th"], **kw)[0]
    except TypeError:
        res = proc.post_process_grounded_object_detection(out, box_threshold=CFG["det_box_th"], **kw)[0]
    labels = res.get("labels") or res.get("text_labels")   # key also moved between versions
    return [dict(label=str(l).strip().lower(), score=float(s), box=[float(v) for v in b])
            for l, s, b in zip(labels, res["scores"], res["boxes"])]

for i, f in enumerate(frames):
    f["dets"] = detect(f["img"])
    print(f"  detecting {i+1}/{len(frames)}", end="\r")

print(f"\n{sum(len(f['dets']) for f in frames)} detections across {len(frames)} frames\n")
hist = defaultdict(int)
for f in frames:
    for d in f["dets"]: hist[d["label"]] += 1
for k, v in sorted(hist.items(), key=lambda x: -x[1]):
    tag = "   <- scale anchor, not inventory" if "door" in k else ""
    print(f"  {k:22} {v}{tag}")
if not any("door" in k for k in hist):
    print("\n  ! No door found. Stage 4 will have nothing to calibrate against.")

In [ ]:
# ============================================================================
# STAGE 2 CHECK · see what the detector actually found
# Teal boxes are doors (the measuring stick). Blue boxes are inventory.
# Look for: things boxed twice, things missed, boxes much bigger than the object.
# ============================================================================

def draw(img, dets):
    v = img.copy()
    for d in dets:
        x0, y0, x1, y1 = [int(z) for z in d["box"]]
        c = (190, 200, 60) if "door" in d["label"] else (240, 120, 40)   # BGR
        cv2.rectangle(v, (x0, y0), (x1, y1), c, 2)
        cv2.putText(v, f"{d['label']} {d['score']:.2f}", (x0, max(14, y0 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1, cv2.LINE_AA)
    return v

n = min(len(frames), 4)
if n:
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.2))
    axes = np.atleast_1d(axes)
    for ax, f in zip(axes, frames[:n]):
        ax.imshow(cv2.cvtColor(draw(f["img"], f["dets"]), cv2.COLOR_BGR2RGB))
        ax.set_title(f"t={f['t']:.1f}s · {len(f['dets'])} objects", fontsize=9); ax.axis("off")
    plt.suptitle("Stage 2 — detections (teal = door / scale anchor)", fontsize=11)
    plt.tight_layout(); plt.show()

## Stage 3 · Depth — how far away is every pixel

**Input:** the keyframes. **Output:** a "point map" giving real-world X, Y, Z in **metres**
for every pixel. **Model:** MoGe-2 from Microsoft.

This is what makes Method B possible at all. Once you know the 3D position of each pixel,
you can measure the object inside a box.

**The catch, and it is the central problem of this whole project.** A photo genuinely does
not know how big things are — a sofa and a doll's sofa produce identical images. MoGe-2
guesses the scale from what it has learned about the world, and its published error is
**8.19%**. Because volume grows with the *cube* of length, an 8% length error becomes
roughly a **26% volume error**.

Worse, it is one mistake applied to everything. If the model thinks the room is 8% bigger
than it is, *every object in it* is 8% too big. Averaging over a hundred objects does not
help, because there is only one error, not a hundred. **Stage 4 exists solely to fix this.**

We use MoGe-2 rather than the better-scoring UniDepthV2 because UniDepthV2 is
non-commercial licensed and could never ship (gap **B2**).

In [ ]:
# ============================================================================
# STAGE 3 · DEPTH — turn a flat picture into 3D points measured in metres
#
# WHY THIS EXISTS
#   Method B cannot measure anything from a flat image. This gives every pixel an
#   (X, Y, Z) position in metres, so an object's size becomes a subtraction.
#
# IN   frames from stage 1
# OUT  f["depth"] on each frame = dict with
#         points     (H, W, 3) — X, Y, Z in metres for each pixel
#         depth      (H, W)    — straight-line distance in metres
#         intrinsics (3, 3)    — the camera geometry the model inferred
#         mask       (H, W)    — True where the model is confident
#
# HOW IT HELPS THE GOAL
#   It is the entire basis of Method B. It is also the biggest single source of
#   error in the POC, which is why stage 4 immediately tries to correct it.
#
# CAUTION
#   We assume the model puts the VERTICAL axis at index 1. Stage 4's door
#   measurement depends on that. The printout below lets you check on frame one.
# ============================================================================

_dep = {}
def depth_load():
    """Load MoGe-2 once and cache it."""
    if "m" not in _dep:
        from moge.model.v2 import MoGeModel
        print(f"loading {CFG['depth_model']} (first run downloads ~1.3GB) ...")
        _dep["m"] = MoGeModel.from_pretrained(CFG["depth_model"]).to(DEVICE).eval()
    return _dep["m"]

def infer_depth(img_bgr):
    """Run MoGe-2 on one frame. Returns numpy arrays, not tensors."""
    model = depth_load()
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    t = torch.tensor(rgb / 255.0, dtype=torch.float32, device=DEVICE).permute(2, 0, 1)
    with torch.no_grad():
        o = model.infer(t)
    return {k: (v.detach().cpu().numpy() if torch.is_tensor(v) else v) for k, v in o.items()}

for i, f in enumerate(frames):
    f["depth"] = infer_depth(f["img"])
    print(f"  depth {i+1}/{len(frames)}", end="\r")

d0  = frames[0]["depth"]
pts = d0["points"]; msk = d0["mask"].astype(bool)
print(f"\nmodel returned: {list(d0.keys())}")
print(f"point map {pts.shape}, {msk.mean()*100:.0f}% of pixels confident\n")
for ax, nm in enumerate("XYZ"):
    v = pts[..., ax][msk]
    print(f"  axis {ax} ({nm}): {v.min():+7.2f} to {v.max():+7.2f} m   span {v.max()-v.min():5.2f} m")

# ---- SANITY CHECK YOU MUST DO ON YOUR FIRST REAL FRAME --------------------
# For a normal room photo the VERTICAL axis should span roughly 2-3m (floor to
# ceiling). If axis 1's span instead looks like the room's WIDTH, change this.
VERT_AXIS = 1
print(f"\nUsing axis {VERT_AXIS} as vertical (floor-to-ceiling) for the door measurement.")
print("If that axis's span above does not look like a room height, change VERT_AXIS.")

In [ ]:
# ============================================================================
# STAGE 3 CHECK · look at the depth map
# Nearby things should be one end of the colour scale and far things the other,
# with object edges roughly where they are in the photo. Large flat patches of one
# colour across different objects mean the depth is unreliable there.
# ============================================================================

f = frames[0]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].imshow(cv2.cvtColor(f["img"], cv2.COLOR_BGR2RGB)); ax[0].set_title("frame"); ax[0].axis("off")
dm = f["depth"]["depth"].copy()
dm[~f["depth"]["mask"].astype(bool)] = np.nan          # grey out low-confidence pixels
im = ax[1].imshow(dm, cmap="viridis")
ax[1].set_title("distance from camera (metres)"); ax[1].axis("off")
plt.colorbar(im, ax=ax[1], fraction=0.035, label="metres")
plt.suptitle("Stage 3 — MoGe-2 depth", fontsize=11)
plt.tight_layout(); plt.show()

## Stage 4 · Scale anchor — teaching the pipeline what a metre is

**Input:** the door boxes from stage 2 + the point maps from stage 3.
**Output:** one number — a correction factor for the whole room.

**This is the most important cell in the notebook.**

A standard UK internal door is **1981 mm** tall. So: find a door, measure how tall stage 3
*thinks* it is, and divide. If the model says the door is 1.80 m, everything it measured is
about 10% too small, and we scale it all up by 1.10.

One detection fixes the one error that averaging cannot touch.

**How to use the switch.** Run the whole notebook twice — `use_scale_anchor = True`, then
`False` — and compare stage 9 both times. That difference tells you whether anchoring is
worth building properly, and it is the finding most likely to change the architecture.

Simulation says the payoff is roughly **one point of class accuracy per point of scale
error** (see the appendix at the end, which runs with no data at all).

In [ ]:
# ============================================================================
# STAGE 4 · SCALE ANCHOR — correct the depth model using an object of known size
#
# WHY THIS EXISTS
#   Stage 3's scale can be ~8% out, which is ~26% out on volume, and it is a single
#   error multiplying the whole room. A UK internal door is a free, reliable ruler
#   present in almost every room photo.
#
# IN   door detections (stage 2) + point maps (stage 3)
# OUT  SCALE — one float. Every measurement in stage 5 is multiplied by it.
#         SCALE > 1  the model was reading things too SMALL
#         SCALE < 1  the model was reading things too BIG
#         SCALE = 1  no correction applied
#
# HOW IT HELPS THE GOAL
#   It attacks the largest error term in the POC for the cost of one detection.
#   Turning it off and re-running is the POC's key experiment.
# ============================================================================

def door_scale(dets, points, mask, vert=VERT_AXIS):
    """Work out a correction factor from the tallest-confidence door in one frame."""
    doors = [d for d in dets if "door" in d["label"]]
    if not doors:
        return None, "no door detected in this frame"
    d = max(doors, key=lambda x: x["score"])                  # most confident door
    x0, y0, x1, y1 = [int(v) for v in d["box"]]
    sub, m = points[y0:y1, x0:x1], mask[y0:y1, x0:x1].astype(bool)
    if m.sum() < 200:
        return None, "door region has too few confident depth pixels"
    v = sub[..., vert][m]
    # 2nd-98th percentile instead of min/max, so a few stray pixels can't ruin it
    measured = float(np.percentile(v, 98) - np.percentile(v, 2))
    if measured < 0.3:
        return None, f"implausible door height {measured:.2f} m — probably a mis-detection"
    return CFG["door_height_m"] / measured, f"door measured as {measured:.3f} m (score {d['score']:.2f})"

factors = []
for f in frames:
    fac, why = door_scale(f["dets"], f["depth"]["points"], f["depth"]["mask"])
    f["scale_factor"], f["scale_why"] = fac, why
    if fac: factors.append(fac)

# Decide which scale to use, in priority order.
if CFG["manual_scale"] is not None:
    SCALE, src = float(CFG["manual_scale"]), "manual override"
elif CFG["use_scale_anchor"] and factors:
    SCALE, src = float(np.median(factors)), f"door anchor, median of {len(factors)} frame(s)"
else:
    SCALE = 1.0
    src = "NONE — raw model scale" + ("" if CFG["use_scale_anchor"] else " (anchor switched OFF)")

print(f"SCALE = {SCALE:.4f}    [{src}]\n")
if factors:
    print(f"  per-frame factors: {', '.join(f'{x:.3f}' for x in factors)}")
    print(f"  agreement spread:  {np.ptp(factors):.3f}   (tight = trustworthy)")
    print(f"\n  Reading: raw depth was about {abs(1-SCALE)*100:.1f}% too "
          f"{'small' if SCALE > 1 else 'large'}.")
    print(f"  Left uncorrected that is roughly {abs(1-SCALE**3)*100:.0f}% error on VOLUME.")
else:
    for f in frames[:3]: print(f"  {f['scale_why']}")
    print("\n  ! No anchor applied. Method B's volumes carry the model's raw scale error.")
    print("    Expect around 26% based on published benchmarks.")

## Stage 5 · Method B — Measure & Compute

**Input:** boxes (stage 2) + point maps (stage 3) + the correction factor (stage 4).
**Output:** width, depth and height in metres for every detected object.

For each box we take all the 3D points inside it and work out how far they spread.

Two details that matter:

- **Height** comes from the vertical axis directly.
- **Width and depth** come from *rotating* the footprint to fit the object. Simply taking
  the camera's left-right spread would be wrong for anything not perfectly square to the
  lens — a sofa at 45° would measure far too wide. We use PCA to find the object's own
  orientation first.

We then also match the measured size to the closest entry in the cube table. That lets us
score Method B two ways: as a *measurer* (are the dimensions right?) and as a *classifier*
(did measuring tell us which size class it is?).

In [ ]:
# ============================================================================
# STAGE 5 · METHOD B — measure each object's real-world size
#
# WHY THIS EXISTS
#   This IS Method B. It answers "how big is this thing" from geometry alone,
#   with no lookup table involved.
#
# IN   boxes (stage 2), point maps (stage 3), SCALE (stage 4)
# OUT  MEAS — a table with one row per measured object:
#         w, d, h        width/depth/height in metres (w >= d by convention)
#         bbox_m3        w * d * h, the raw box volume
#         mapped_class   nearest entry in the cube table
#         map_dist       how close that match was (0 = perfect, lower is better)
#
# HOW IT HELPS THE GOAL
#   Produces the "measure it" half of the race. If these dimensions turn out
#   accurate, geometry is viable; if not, we stop investing in it.
# ============================================================================

def extent(points, mask, box, scale=1.0, vert=VERT_AXIS, lo=2, hi=98):
    """Real-world width/depth/height of whatever is inside `box`."""
    x0, y0, x1, y1 = [int(v) for v in box]
    sub, m = points[y0:y1, x0:x1], mask[y0:y1, x0:x1].astype(bool)
    if m.sum() < 100:
        return None                                   # not enough confident depth here
    p = sub[m] * scale                                # apply stage 4's correction

    # Height: straight off the vertical axis. Percentiles reject stray edge pixels.
    h = float(np.percentile(p[:, vert], hi) - np.percentile(p[:, vert], lo))

    # Footprint: rotate to the object's own axes before measuring, otherwise an
    # object sitting at an angle to the camera measures much too wide.
    horiz = np.delete(p, vert, axis=1)                # drop the vertical column
    horiz = horiz - horiz.mean(0)                     # centre it
    try:
        _, _, vt = np.linalg.svd(horiz, full_matrices=False)   # PCA
        proj = horiz @ vt.T                           # now axis 0 is the object's long side
    except np.linalg.LinAlgError:
        proj = horiz                                  # degenerate: fall back to camera axes

    w = float(np.percentile(proj[:, 0], hi) - np.percentile(proj[:, 0], lo))
    d = float(np.percentile(proj[:, 1], hi) - np.percentile(proj[:, 1], lo))
    w, d = max(w, d), min(w, d)                       # convention: width is the longer side
    return dict(w=abs(w), d=abs(d), h=abs(h), bbox_m3=abs(w*d*h), n_pts=int(m.sum()))

def nearest_class(dims, allowed=None):
    """Which cube-table entry do these measurements look most like?

    `allowed` restricts the candidates to what the detector label permits. That
    restriction is worth ~18 points of accuracy at realistic error levels — never
    search the whole table (see the appendix).
    """
    tgt = np.sort([dims["w"], dims["d"], dims["h"]])[::-1]   # sort so orientation can't matter
    best, bd = None, 1e9
    for name in (allowed or CLASS_NAMES):
        c = CLASSES.get(name)
        if not c: continue
        cand = np.sort(c["typical_dims_m"])[::-1]
        dist = float(np.linalg.norm(tgt - cand) / max(np.linalg.norm(cand), 1e-6))
        if dist < bd: best, bd = name, dist
    return best, bd

rows = []
for f in frames:
    for d in f["dets"]:
        if "door" in d["label"]:
            continue                                  # doors are the ruler, not cargo
        e = extent(f["depth"]["points"], f["depth"]["mask"], d["box"], scale=SCALE)
        if not e:
            continue
        allowed = VOCAB.get(d["label"]) or None       # narrow candidates by what was detected
        cls, dist = nearest_class(e, allowed)
        rows.append(dict(t=f["t"], det_label=d["label"], score=d["score"], **e,
                         mapped_class=cls, map_dist=round(dist, 3),
                         cube_m3=CLASSES[cls]["cube_m3"] if cls else np.nan))

MEAS = pd.DataFrame(rows)
if len(MEAS):
    print(f"Method B measured {len(MEAS)} objects across {len(frames)} frames\n")
    print(MEAS[["t", "det_label", "w", "d", "h", "bbox_m3", "mapped_class", "map_dist"]]
          .round(3).head(20).to_string(index=False))
    print("\n  map_dist above ~0.25 means the measurement did not look much like ANY")
    print("  size class — treat those rows with suspicion.")
else:
    print("Method B measured nothing. Check stage 2 found boxes and stage 3's mask coverage.")

## Stage 6 · Method A — Recognise & Look Up

**Input:** the keyframes. **Output:** a list of size classes with counts.
**Model:** Claude Sonnet 5 (vision).

This is the surveyor-with-a-clipboard approach, and it measures nothing at all.

The trick worth understanding — and it is how the commercial products do it — is that they
distinguish *"queen mattress vs king mattress"* and *"two-seater vs sectional"*. They are
**not measuring those**. They are **choosing between named categories, and the category
carries the volume.** Picking one of three sofa classes is a far easier problem than
getting three continuous numbers right.

We force the answer through a tool schema whose allowed values are exactly our cube table,
so the model physically cannot invent a class we have no volume for.

In [ ]:
# ============================================================================
# STAGE 6 · METHOD A — recognise each object and look its volume up
#
# WHY THIS EXISTS
#   This IS Method A. No measuring: name the thing precisely enough that a standard
#   table gives you its packed volume. Every product on the market works this way.
#
# IN   frames from stage 1 (up to CFG['vlm_max_frames'], for cost control)
# OUT  per_frame_recognised — one inventory per frame:
#         {room_type, t, items: [{size_class, count, confidence, reasoning}]}
#
# HOW IT HELPS THE GOAL
#   Produces the "recognise it" half of the race. It also tests something specific:
#   can a model reliably choose between SIMILAR size classes (double vs king bed),
#   which is where all the volume difference lives.
#
# NOTE ON COST
#   Roughly $0.02 per frame on Sonnet 5. Six frames is about $0.12 a room.
# ============================================================================

import base64

METHOD_A_PROMPT = """You are cataloguing a room for a household removal survey.

List every MOVABLE item you can see. For each one pick the single closest size_class from
the allowed list — the class carries the packed volume, so choosing between e.g.
sofa_2_seat / sofa_3_seat / sofa_sectional matters as much as recognising it is a sofa.

Rules:
- Count only what is visible IN THIS IMAGE. Do not infer items you cannot see.
- Do not count fitted or structural things: fitted kitchen units, built-in wardrobes,
  radiators, doors, windows, flooring, light fittings.
- If an item is partly hidden, still count it once and say so in reasoning.
- If unsure between two size classes, pick the smaller and lower your confidence.
- confidence is 0.0-1.0 for the size_class choice, not for the item existing."""

def classify_frame(img_bgr, model=None):
    """Ask the vision model for a structured inventory of one frame."""
    from anthropic import Anthropic
    client = Anthropic()
    ok, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    b64 = base64.b64encode(buf.tobytes()).decode()

    # A forced tool call is how we guarantee machine-readable output. The `enum`
    # is our cube table, so an unknown class is impossible by construction.
    tool = {
        "name": "record_inventory",
        "description": "Record every movable household item visible in this room image.",
        "input_schema": {
            "type": "object",
            "properties": {
                "room_type": {"type": "string", "description": "bedroom, living, kitchen, other"},
                "items": {"type": "array", "items": {
                    "type": "object",
                    "properties": {
                        "size_class": {"type": "string", "enum": CLASS_NAMES},
                        "count":      {"type": "integer", "minimum": 1},
                        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                        "reasoning":  {"type": "string"},
                    },
                    "required": ["size_class", "count", "confidence"],
                }},
            },
            "required": ["room_type", "items"],
        },
    }
    r = client.messages.create(
        model=model or CFG["vlm_model"], max_tokens=2048,
        tools=[tool], tool_choice={"type": "tool", "name": "record_inventory"},
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
            {"type": "text", "text": METHOD_A_PROMPT},
        ]}],
    )
    for blk in r.content:
        if blk.type == "tool_use":
            return blk.input, r.usage
    return {"room_type": "unknown", "items": []}, r.usage

per_frame_recognised, usage_tot = [], dict(input=0, output=0)
if HAS_KEY:
    for f in frames[: CFG["vlm_max_frames"]]:
        inv, u = classify_frame(f["img"])
        inv["t"] = f["t"]
        per_frame_recognised.append(inv)
        usage_tot["input"] += u.input_tokens; usage_tot["output"] += u.output_tokens
        items = ", ".join(f"{i['count']}x {i['size_class']}" for i in inv["items"]) or "(nothing)"
        print(f"t={f['t']:5.1f}s  {inv['room_type']:8}  {items}")
    # Sonnet 5 list price: $2 per Mtok in, $10 per Mtok out
    cost = usage_tot["input"] / 1e6 * 2.0 + usage_tot["output"] / 1e6 * 10.0
    print(f"\ntokens in={usage_tot['input']:,} out={usage_tot['output']:,}   cost ~${cost:.3f} for this room")
else:
    print("No ANTHROPIC_API_KEY — Method A skipped.")
    print("Method B results are still valid; stage 9 will only score Method B.")

## Stage 7 · De-duplication — one sofa, not twenty

**Input:** the per-frame inventories from both methods. **Output:** one inventory per method.

The same sofa appears in every frame. Naive pipelines add them up and report twenty sofas.
This is the single most common way a pipeline like this produces a wildly wrong number.

Two blunt rules, neither perfect:

| Rule | Logic | Survives | Breaks on |
|------|-------|----------|-----------|
| `max` | trust the frame that saw the most | things being hidden in other frames | one false positive inflating the count |
| `median` | trust the typical frame | the model imagining something once | genuinely crowded rooms where most frames see less |

Proper object tracking between frames is the real fix. This is enough for a first reading.

In [ ]:
# ============================================================================
# STAGE 7 · DEDUPLICATION — collapse per-frame lists into one room inventory
#
# WHY THIS EXISTS
#   Frames overlap. Without this step every object is counted once per frame it
#   appears in, and the volume comes out several times too large.
#
# IN   per_frame_recognised (Method A) and MEAS (Method B)
# OUT  inv_recognised, inv_measured — dicts of {size_class: count}
#
# HOW IT HELPS THE GOAL
#   Counts multiply straight into volume, so this is as important as measuring
#   well. It is also gap B1 — a known weak point we are choosing to accept for now.
# ============================================================================

def dedup_counts(per_frame_invs, rule="max"):
    """One count per size class, from many per-frame counts."""
    if not per_frame_invs:
        return {}
    per = []
    for inv in per_frame_invs:
        c = defaultdict(int)
        for it in inv["items"]:
            c[it["size_class"]] += int(it["count"])
        per.append(c)
    keys = sorted({k for c in per for k in c})
    out = {}
    for k in keys:
        # Zeros are included deliberately: a class seen in only 1 of 8 frames gets a
        # median of 0 and is dropped, which is the false-positive filter.
        series = [c.get(k, 0) for c in per]
        v = max(series) if rule == "max" else int(np.median(series))
        if v > 0:
            out[k] = v
    return out

# --- Method A: dedup the model's per-frame inventories ---
inv_recognised = dedup_counts(per_frame_recognised, CFG["dedup_rule"])

# --- Method B: count how many of each class were measured in a single frame, take the max.
#     Same idea, but Method B has one row per detection rather than a count.
inv_measured = {}
if len(MEAS):
    per_frame_counts = defaultdict(lambda: defaultdict(int))
    for r in MEAS.itertuples():
        if isinstance(r.mapped_class, str):
            per_frame_counts[r.t][r.mapped_class] += 1
    for cls in {c for d in per_frame_counts.values() for c in d}:
        inv_measured[cls] = max(d.get(cls, 0) for d in per_frame_counts.values())

print(f"METHOD A · Recognise & Look Up   (rule='{CFG['dedup_rule']}', {len(per_frame_recognised)} frames)")
for k, v in sorted(inv_recognised.items()): print(f"    {v} x {k}")
if not inv_recognised: print("    (nothing — method skipped or found nothing)")

print(f"\nMETHOD B · Measure & Compute     (max per frame, {len(frames)} frames)")
for k, v in sorted(inv_measured.items()): print(f"    {v} x {k}")
if not inv_measured: print("    (nothing)")

only_a = set(inv_recognised) - set(inv_measured)
only_b = set(inv_measured) - set(inv_recognised)
if only_a: print(f"\n  only Method A saw: {sorted(only_a)}")
if only_b: print(f"  only Method B saw: {sorted(only_b)}")

## Stage 8 · Aggregate — three volume figures

**Input:** both inventories + the cube table. **Output:** three numbers to compare.

| Figure | How | Tests |
|--------|-----|-------|
| Method A | recognised class → table | can it *name* things well enough? |
| Method B mapped | measured size → nearest class → table | can measuring *identify* things? |
| Method B raw | measured w x d x h added up | can it *measure* things? |

The raw figure should come out **lower** than the table figures, and that is correct, not a
bug. A cube sheet holds **packed** volume — padding, crating, and the air you inevitably
lose stacking a truck. A tape measure round a sofa does not capture any of that (gap **B4**).

In [ ]:
# ============================================================================
# STAGE 8 · AGGREGATE — turn inventories into the number the business cares about
#
# WHY THIS EXISTS
#   "How much space will this take" is the actual question. Everything before this
#   was machinery; this is the answer, computed three ways so they can be compared.
#
# IN   inv_recognised, inv_measured, MEAS, cube_table.json
# OUT  a small table of volumes in m3 and ft3
#
# HOW IT HELPS THE GOAL
#   These three numbers, set against the hand measurements in stage 9, are the
#   entire deliverable of the POC.
# ============================================================================

def vol_from_inventory(inv):
    """Sum packed volume for an inventory using the cube table."""
    return float(sum(inv[c] * CLASSES[c]["cube_m3"] for c in inv if c in CLASSES))

vol_recognised     = vol_from_inventory(inv_recognised)              # Method A
vol_measured_class = vol_from_inventory(inv_measured)                # Method B, via lookup
vol_measured_raw   = float(MEAS["bbox_m3"].sum()) if len(MEAS) else 0.0   # Method B, pure geometry

M3_TO_FT3 = 1 / 0.0283168
res = pd.DataFrame([
    ("A · recognise -> table",        vol_recognised,     len(inv_recognised)),
    ("B · measure -> class -> table", vol_measured_class, len(inv_measured)),
    ("B · measure -> raw box volume", vol_measured_raw,   len(MEAS)),
], columns=["method", "volume_m3", "n_items"])
res["volume_ft3"] = res["volume_m3"] * M3_TO_FT3

print(res.round(2).to_string(index=False))
print(f"\nscale correction applied: {SCALE:.4f}   [{src}]")
print("\nExpect the raw box figure to be LOWER than the table figures. Cube sheets carry")
print("PACKED volume (padding, crating, stacking voids); a tape measure does not (gap B4).")

## Stage 9 · Evaluate — the only cell that answers the question

**Input:** `ground_truth.csv` (your hand measurements). **Output:** how wrong each method is.

Needs the file to exist. Copy `ground_truth_template.csv`, measure the same rooms with a
laser measure, fill it in.

**Read `vol_bias_pct` before `vol_abs_err_pct`.** They need different fixes:

- **Bias** — consistently 15% low. Fixable with a single multiplier. Almost good news.
- **Spread** — randomly ±15%. Not fixable with a coefficient. The harder problem.

A method with big bias and small spread is *better* than the reverse, even if its raw
error looks worse.

In [ ]:
# ============================================================================
# STAGE 9 · EVALUATE — score both methods against hand measurements
#
# WHY THIS EXISTS
#   Without this the notebook is a demo. This is what makes it an experiment.
#
# IN   ground_truth.csv — one row per real item, with measured dimensions
# OUT  precision/recall on the item list, volume bias and absolute error per
#      method, per-item dimension error for Method B, and a "missed entirely" list
#
# HOW IT HELPS THE GOAL
#   Answers the three questions the POC exists to answer:
#     1. how wrong is each method
#     2. does the scale anchor help (re-run with it off and compare)
#     3. which method should we keep
# ============================================================================

GT_PATH = ROOT / "ground_truth.csv"
if not GT_PATH.exists():
    print(f"No {GT_PATH.name} — nothing to score against.")
    print("Copy ground_truth_template.csv, measure the room, fill it in, re-run. Gap A1.")
else:
    gt = pd.read_csv(GT_PATH, comment="#")
    room = CFG["score_room"] or gt.room_id.iloc[0]
    print(f"rooms available: {sorted(gt.room_id.unique())}   scoring: {room}")
    g = gt[gt.room_id == room]

    true_counts = g.groupby("true_class")["qty"].sum().to_dict()
    true_vol    = sum(n * CLASSES[c]["cube_m3"] for c, n in true_counts.items() if c in CLASSES)

    def score(pred, name):
        """Compare a predicted inventory against the truth."""
        cls = set(pred) | set(true_counts)
        # true positives = overlap in counts, class by class
        tp = sum(min(pred.get(c, 0), true_counts.get(c, 0)) for c in cls)
        pc, tc = sum(pred.values()), sum(true_counts.values())
        pv = vol_from_inventory(pred)
        return dict(
            method=name,
            precision=round(tp / pc, 3) if pc else 0.0,   # of what we claimed, how much was real
            recall=round(tp / tc, 3) if tc else 0.0,      # of what was there, how much we found
            pred_items=pc, true_items=tc,
            pred_m3=round(pv, 3), true_m3=round(true_vol, 3),
            vol_bias_pct=round((pv - true_vol) / true_vol * 100, 1) if true_vol else np.nan,
            vol_abs_err_pct=round(abs(pv - true_vol) / true_vol * 100, 1) if true_vol else np.nan)

    print(f"\n=== ROOM {room} ===")
    scored = []
    if inv_recognised: scored.append(score(inv_recognised, "A recognise"))
    else:              print("(Method A skipped — not scored)")
    if inv_measured:   scored.append(score(inv_measured,   "B measure"))
    else:              print("(Method B found nothing — check stages 2 and 3)")
    if scored:
        print(pd.DataFrame(scored).to_string(index=False))

    # --- Method B only: how accurate were the actual dimensions? ---
    if len(MEAS):
        dim_rows = []
        for _, r in g.iterrows():
            m = MEAS[MEAS.mapped_class == r.true_class]
            if not len(m): continue
            tgt_v = r.width_m * r.depth_m * r.height_m
            m = m.loc[(m["bbox_m3"] - tgt_v).abs().idxmin()]      # closest matching detection
            truth = np.sort([r.width_m, r.depth_m, r.height_m])[::-1]
            meas  = np.sort([m.w, m.d, m.h])[::-1]
            dim_rows.append(dict(item=r.true_class,
                                 true_dims=np.round(truth, 2).tolist(),
                                 meas_dims=np.round(meas, 2).tolist(),
                                 dim_err_pct=round(float(np.mean(np.abs(meas - truth) / truth)) * 100, 1)))
        if dim_rows:
            D = pd.DataFrame(dim_rows)
            print("\n--- Method B: dimension accuracy ---")
            print(D.to_string(index=False))
            mean_err = D.dim_err_pct.mean()
            print(f"\nmean dimension error {mean_err:.1f}%")
            print(f"  -> roughly {mean_err*3:.0f}% on volume, since volume goes as length cubed")

    missed = {c: n for c, n in true_counts.items()
              if c not in inv_recognised and c not in inv_measured}
    if missed:
        print(f"\nMissed entirely by BOTH methods: {missed}")
        print("  These are detector vocabulary gaps — cheap to fix in detect_vocab.json.")

## What to do with these results

1. **Bias before absolute error.** Consistent bias is one multiplier away from fixed.
   Random spread is the expensive problem.
2. **Re-run with `use_scale_anchor = False`** and diff the two stage 9 outputs. That number
   decides whether anchoring gets built properly.
3. **Compare Method A against Method B.** If recognise-and-look-up wins — as the commercial
   evidence suggests — stop building geometry and put the effort into the size-class
   taxonomy and getting NX's real cube table instead.
4. **Check the "missed entirely" list.** Pure vocabulary gaps, cheap to close.
5. **Then widen.** Two room types across 3-5 properties before believing anything (gap B7).

Open questions are all tracked in `../GAPS.md`; current build state in `../STATUS.md`.

## Appendix · How accurate does the scale have to be?

**Runs with no input data at all.** Simulates measurement error and reports how often
Method B still picks the right size class, so you know what to expect before you have
footage — and so you can tell a bad result from a bad *setup*.

It applies a **whole-room scale error** (the way the depth model's error actually behaves)
plus small per-object noise, then checks class assignment with and without the detector
narrowing the candidates.

In [ ]:
# ============================================================================
# APPENDIX · SENSITIVITY — what precision does Method B actually need?
#
# WHY THIS EXISTS
#   Tells you the answer before you collect data, so stage 9's output can be read
#   in context. Also proves a specific design rule: never match a measurement
#   against the whole cube table.
#
# IN   nothing but the cube table
# OUT  printed tables: which class pairs are hardest to tell apart, and class
#      accuracy at 0 / 5 / 8 / 15% scale error
#
# HOW IT HELPS THE GOAL
#   Turns "is measuring viable" from an opinion into a number, at zero cost.
# ============================================================================

import math, random

def _nearest_ranked(dims, allowed=None):
    """Every candidate class ranked by how well it matches `dims`. Closest first."""
    tgt = sorted(dims, reverse=True); out = []
    for name in (allowed or CLASS_NAMES):
        c = CLASSES.get(name)
        if not c: continue
        cand = sorted(c["typical_dims_m"], reverse=True)
        nrm = math.sqrt(sum(x*x for x in cand)) or 1e-6
        out.append((math.sqrt(sum((a-b)**2 for a, b in zip(tgt, cand)))/nrm, name))
    out.sort(); return out

# --- which size classes are genuinely hard to separate on size alone? ---
print("Hardest pairs to tell apart by measurement (smaller margin = more fragile):\n")
margins = []
for prompt, cands in VOCAB.items():
    if len(cands) < 2: continue
    for truth in cands:
        r = _nearest_ranked(CLASSES[truth]["typical_dims_m"], cands)
        margins.append((r[1][0] - r[0][0], truth, r[1][1]))
for m, a, b in sorted(margins)[:6]:
    print(f"  {a:22} vs {b:22} margin {m:.3f}")

# --- how fast does class accuracy fall as scale error grows? ---
random.seed(7)
GROUPS = [k for k, v in VOCAB.items() if len(v) > 1]
print(f"\nClass accuracy vs scale error — 2000 trials per row, {len(GROUPS)} ambiguous groups:\n")
print(f"  {'scale err':>10} {'detector-narrowed':>19} {'whole table':>13}")
for se in [0.00, 0.05, 0.08, 0.15]:
    okr = oku = n = 0
    for _ in range(2000):
        cands = VOCAB[random.choice(GROUPS)]
        truth = random.choice(cands)
        sc = 1.0 + random.gauss(0, se)                       # one error for the whole room
        dims = [d * sc * (1 + random.gauss(0, 0.03)) for d in CLASSES[truth]["typical_dims_m"]]
        okr += _nearest_ranked(dims, cands)[0][1] == truth   # candidates restricted
        oku += _nearest_ranked(dims)[0][1] == truth          # searching everything
        n += 1
    if se == 0.08: gap8 = (okr - oku) / n * 100
    flag = "  <- MoGe-2 published error" if se == 0.08 else ""
    print(f"  {se*100:9.0f}% {okr/n*100:18.1f}% {oku/n*100:12.1f}%{flag}")

print(f"""
TWO CONCLUSIONS

1. Always narrow candidates by the detector label. At the depth model's real-world
   8% error that restriction is worth about {gap8:.0f} percentage points of class
   accuracy. Matching a measurement against all {len(CLASS_NAMES)} classes is a mistake.

2. Roughly one point of class accuracy is lost per point of scale error. That is the
   measurable payoff from the stage 4 anchor, and why it outranks picking a better
   depth model. The pairs listed above go wrong first — for those, asking the customer
   one question beats any amount of better geometry.
""")